# Regression Model - Predict Actual Speedup

Instead of binary classification (beneficial/not), predict the actual speedup value.

**Why regression is better:**
- Captures full information (1.26x >> 1.06x, both were just "beneficial")
- Can rank loops by predicted benefit
- More useful for real compiler decisions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

project_root = Path.cwd().parent

## Load Data

In [ ]:
df = pd.read_csv(project_root / 'data' / 'processed' / 'dataset.csv')
print(f"Dataset: {len(df)} loops")
print(f"\nSpeedup distribution:")
print(df['speedup'].describe())

# visualize target
plt.figure(figsize=(10, 4))
plt.hist(df['speedup'], bins=30, edgecolor='black', alpha=0.7)
plt.axvline(1.0, color='red', linestyle='--', label='No benefit')
plt.axvline(df['speedup'].mean(), color='green', linestyle='--', label=f'Mean: {df["speedup"].mean():.3f}')
plt.xlabel('Speedup')
plt.ylabel('Count')
plt.title('Target Variable Distribution (Speedup)')
plt.legend()
plt.show()

## Feature Engineering

In [ ]:
# add engineered features
df['memory_ops'] = df['num_load_instructions'] + df['num_store_instructions']
df['memory_ratio'] = df['memory_ops'] / (df['num_instructions'] + 1)
df['compute_ratio'] = df['num_arithmetic_ops'] / (df['num_instructions'] + 1)
df['branch_ratio'] = df['num_branches'] / (df['num_instructions'] + 1)
df['phi_ratio'] = df['num_phi_nodes'] / (df['num_instructions'] + 1)
df['trip_count_log'] = np.log10(df['estimated_trip_count'].replace(-1, 1000000) + 1)
df['instructions_per_bb'] = df['num_instructions'] / (df['num_basic_blocks'] + 1)
df['has_calls'] = (df['num_calls'] > 0).astype(int)
df['deps_per_memop'] = df['num_memory_dependencies'] / (df['memory_ops'] + 1)

print("Features created")

In [ ]:
# select features
feature_cols = [
    'num_instructions', 'num_basic_blocks', 'num_load_instructions',
    'num_store_instructions', 'num_branches', 'num_calls', 'num_arithmetic_ops',
    'estimated_trip_count', 'has_constant_trip_count', 'nesting_depth',
    'num_phi_nodes', 'num_memory_dependencies', 'has_early_exit', 'num_exits',
    'memory_ratio', 'compute_ratio', 'branch_ratio', 'phi_ratio',
    'trip_count_log', 'instructions_per_bb', 'has_calls', 'deps_per_memop'
]

X = df[feature_cols].copy()
y = df['speedup'].copy()  # REGRESSION TARGET

X['estimated_trip_count'] = X['estimated_trip_count'].replace(-1, 1000000)
X = X.fillna(0)

print(f"Features: {len(feature_cols)}")
print(f"Target range: {y.min():.3f} - {y.max():.3f}")

## Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")

## Train Regression Models

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0, random_state=42),
    'Decision Tree': DecisionTreeRegressor(random_state=42, max_depth=5),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10),
}

results = {}

print("="*70)
print("Regression Model Performance")
print("="*70)

for name, model in models.items():
    print(f"\n{name}:")
    
    # use scaled for linear models, unscaled for trees
    if 'Linear' in name or 'Ridge' in name or 'Lasso' in name:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print(f"  MAE (avg error):  {mae:.4f}")
    print(f"  RMSE:             {rmse:.4f}")
    print(f"  R² score:         {r2:.4f}")
    
    results[name] = {
        'model': model,
        'predictions': y_pred,
        'mae': mae,
        'rmse': rmse,
        'r2': r2
    }

print("\n" + "="*70)

## Prediction vs Actual

In [ ]:
# scatter plots: predicted vs actual
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, (name, result) in enumerate(results.items()):
    ax = axes[idx]
    y_pred = result['predictions']
    
    ax.scatter(y_test, y_pred, alpha=0.6)
    
    # perfect prediction line
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect prediction')
    
    ax.set_xlabel('Actual Speedup')
    ax.set_ylabel('Predicted Speedup')
    ax.set_title(f'{name}\nMAE: {result["mae"]:.4f}, R²: {result["r2"]:.4f}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Residual Analysis

In [ ]:
# residuals (errors) for best model
best_name = max(results.keys(), key=lambda k: results[k]['r2'])
best_result = results[best_name]
residuals = y_test - best_result['predictions']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# residual plot
ax1.scatter(best_result['predictions'], residuals, alpha=0.6)
ax1.axhline(0, color='red', linestyle='--')
ax1.set_xlabel('Predicted Speedup')
ax1.set_ylabel('Residual (Actual - Predicted)')
ax1.set_title(f'Residual Plot ({best_name})')
ax1.grid(True, alpha=0.3)

# residual distribution
ax2.hist(residuals, bins=15, edgecolor='black', alpha=0.7)
ax2.axvline(0, color='red', linestyle='--', label='Zero error')
ax2.axvline(residuals.mean(), color='green', linestyle='--', label=f'Mean: {residuals.mean():.4f}')
ax2.set_xlabel('Residual')
ax2.set_ylabel('Count')
ax2.set_title('Residual Distribution')
ax2.legend()

plt.tight_layout()
plt.show()

print(f"Best model: {best_name}")
print(f"Mean error: {residuals.mean():.4f}")
print(f"Std error: {residuals.std():.4f}")

## Feature Importance

In [ ]:
# random forest feature importances
rf_model = results['Random Forest']['model']
importances = rf_model.feature_importances_

feature_imp = pd.DataFrame({
    'feature': feature_cols,
    'importance': importances
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
top_n = 15
plt.barh(feature_imp['feature'][:top_n], feature_imp['importance'][:top_n])
plt.xlabel('Importance')
plt.title(f'Top {top_n} Features (Random Forest Regressor)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("Top 10 features:")
print(feature_imp.head(10))

## Decision Making with Regression

How to use regression predictions in practice:
1. Predict speedup for all loops
2. Unroll if predicted speedup > threshold (e.g., 1.05)
3. Rank loops by predicted benefit (focus on high-impact ones)

In [ ]:
# simulate decision making
threshold = 1.05
y_pred = best_result['predictions']

# decisions based on predictions
pred_decisions = (y_pred > threshold).astype(int)
actual_decisions = (y_test > threshold).astype(int)

# accuracy of binary decisions derived from regression
from sklearn.metrics import accuracy_score, confusion_matrix
acc = accuracy_score(actual_decisions, pred_decisions)
cm = confusion_matrix(actual_decisions, pred_decisions)

print(f"Decision accuracy (threshold {threshold}): {acc:.3f}")
print(f"\nConfusion matrix:")
print(cm)

# but the REAL value is ranking
test_results = pd.DataFrame({
    'actual_speedup': y_test.values,
    'predicted_speedup': y_pred,
    'error': np.abs(y_test.values - y_pred)
}).sort_values('predicted_speedup', ascending=False)

print("\nTop 5 predicted opportunities:")
print(test_results.head())

print("\nWorst predictions (biggest errors):")
print(test_results.sort_values('error', ascending=False).head())

## Save Model

In [ ]:
import pickle

model_dir = project_root / 'models'
model_dir.mkdir(exist_ok=True)

# save best model
with open(model_dir / 'regression_model.pkl', 'wb') as f:
    pickle.dump(best_result['model'], f)

with open(model_dir / 'scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open(model_dir / 'feature_names.txt', 'w') as f:
    f.write('\n'.join(feature_cols))

print(f"✓ Saved {best_name} regression model")
print(f"  MAE: {best_result['mae']:.4f}")
print(f"  R²: {best_result['r2']:.4f}")
print(f"\nInterpretation:")
print(f"  On average, predictions are off by {best_result['mae']:.4f}x speedup")
print(f"  Model explains {best_result['r2']*100:.1f}% of variance in speedup")

## Summary

**Regression vs Classification:**
- Captures full speedup information (not just binary)
- Can rank loops by predicted benefit
- More useful for compiler decisions

**Next steps:**
1. Compare predictions vs LLVM's decisions
2. Find loops where model disagrees with LLVM
3. Collect more data to improve predictions